# Practical: from regression to a neural network
### Advanced Remote Sensing · Introduction to Deep Learning

In the lecture we built a neural network step by step, starting from a simple linear regression.
Today you do the same thing in code, with the same example: **estimating leaf area index (LAI) from Sentinel-2 reflectance.**

| Part | What you do | Time |
|---|---|---|
| 0 | Setup and load the data | 5 min |
| 1 | Look at the data | 10 min |
| 2 | Linear regression: one band, then many bands | 15 min |
| 3 | The same regression as a single neuron | 15 min |
| 4 | A small neural network | 20 min |
| 5 | Overfitting | 10 min |
| 6 | Random vs. spatial validation | 15 min |

**How to work with this notebook**
* Run a cell with **Shift + Enter**. Run the cells in order, from top to bottom.
* ✏️ marks a **task**: change a number or a line in the code and run the cell again.
* 💬 marks a **question**: think about it or discuss it with your neighbour. Click *Show answer* to check.
* You don't need to write code from scratch. Read the code, change the marked parts, and look at what happens.
* If something breaks: *Runtime → Restart session*, then run all cells from the top again.

## Part 0 · Setup

First we load the Python packages we need. All of them are already installed in Google Colab.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from sklearn.linear_model import LinearRegression

# fixed random seeds, so everyone gets (almost) the same results
np.random.seed(42)
torch.manual_seed(42)

# colours from the lecture slides
DATA, MODEL, HIGHLIGHT = "#1F6F8B", "#50164A", "#B8924A"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("Ready! PyTorch version:", torch.__version__)

Now we load the data. It is a CSV file with one row per pixel.

**Option A:** your lecturer gave you a link. Paste it into `DATA_URL` below.
**Option B:** you have the file `lai_s2_landscape.csv`. Run the cell and upload it when the upload button appears.

In [ ]:
DATA_URL = ""                      # paste the link to the CSV file here (optional)
FILE = "lai_s2_landscape.csv"

if os.path.exists(FILE):
    df = pd.read_csv(FILE)
elif DATA_URL:
    df = pd.read_csv(DATA_URL)
else:
    from google.colab import files  # only works in Google Colab
    files.upload()
    df = pd.read_csv(FILE)

print(f"{len(df)} pixels loaded")
df.head()

**What is in the table?**
* `x`, `y`: position of the pixel in our study area (a grid of 120 × 120 pixels, 20 m each).
* `B2` … `B12`: Sentinel-2 surface reflectance in 10 bands (B4 = red, B5 = red edge, B8 = NIR, B11 = SWIR, …).
* `NDVI`: computed from B8 and B4.
* `LAI`: the leaf area index we want to predict (m² of leaves per m² of ground).

The data are **simulated** with the radiative transfer model PROSAIL, just like in the lecture.
Leaf, canopy and soil properties change smoothly across the study area, so neighbouring pixels are similar, as in a real image.
Think of the LAI values as field measurements we have for these 1500 pixels.

## Part 1 · Look at the data  ⏱ ~10 min

Always look at your data before you model it. First: where are the pixels, and how does LAI change across the area?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, col in zip(axes, ["LAI", "NDVI"]):
    sc = ax.scatter(df.x, df.y, c=df[col], s=8, cmap="YlGn")
    plt.colorbar(sc, ax=ax, label=col)
    ax.set_title(f"{col} of the 1500 pixels")
    ax.set_xlabel("x (pixels)"); ax.set_ylabel("y (pixels)"); ax.set_aspect("equal")
plt.tight_layout(); plt.show()

Next: how does LAI relate to NIR reflectance and to NDVI?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col in zip(axes, ["B8", "NDVI"]):
    ax.scatter(df[col], df.LAI, s=6, alpha=0.5, color=DATA)
    ax.set_xlabel(col); ax.set_ylabel("LAI")
axes[0].set_title("LAI vs. NIR reflectance (B8)"); axes[1].set_title("LAI vs. NDVI")
plt.tight_layout(); plt.show()

💬 **Questions**
1. Look at the maps. Are neighbouring pixels similar or random? Why could that matter later?
2. Which relationship looks more like a straight line: LAI vs. B8 or LAI vs. NDVI?

<details>
<summary><b>Show answer</b></summary>

1. Neighbouring pixels are very similar: LAI forms smooth patches. This is called *spatial autocorrelation*. It will matter in Part 6, when we test our models.
2. LAI vs. B8 is roughly linear (with a lot of scatter). LAI vs. NDVI is clearly curved: NDVI saturates at high LAI. A straight line will not work well there.

</details>

## Part 2 · Linear regression  ⏱ ~15 min

We start with what you know: a straight line, $\hat{y} = w \cdot x + b$.

But first we put some pixels aside. We **train** the model on 70 % of the pixels and **test** it on the other 30 %, which the model never sees during training.
For now we pick the test pixels at random. (We will question this choice in Part 6.)

In [ ]:
rng = np.random.default_rng(0)
is_test = rng.random(len(df)) < 0.3          # 30 % of the pixels, chosen at random
train, test = df[~is_test], df[is_test]
print(f"training pixels: {len(train)},  test pixels: {len(test)}")


def evaluate(y_true, y_pred, label=""):
    """Print RMSE and R² of a prediction."""
    y_true, y_pred = np.asarray(y_true).ravel(), np.asarray(y_pred).ravel()
    rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
    r2 = 1 - np.sum((y_pred - y_true) ** 2) / np.sum((y_true - y_true.mean()) ** 2)
    print(f"{label:<28} RMSE = {rmse:.2f}   R² = {r2:.2f}")
    return rmse, r2

**RMSE** (root mean squared error) is the typical size of the error, in LAI units. Smaller is better.
**R²** says how much of the variation in LAI the model explains. 1 is perfect, 0 is useless.

Now we fit a line with one band, NIR (B8):

In [ ]:
bands = ["B8"]                                 # ✏️ the input band(s)

linreg = LinearRegression().fit(train[bands], train.LAI)
print("weights w:", np.round(linreg.coef_, 2), "   bias b:", round(linreg.intercept_, 2))
evaluate(test.LAI, linreg.predict(test[bands]), f"Linear regression {bands}")

if len(bands) == 1:                           # we can only draw a line for one input band
    xs = np.linspace(df[bands[0]].min(), df[bands[0]].max(), 50)
    plt.scatter(train[bands[0]], train.LAI, s=6, alpha=0.4, color=DATA, label="training pixels")
    plt.plot(xs, linreg.predict(pd.DataFrame({bands[0]: xs})), color=MODEL, lw=3, label="regression line")
    plt.xlabel(bands[0]); plt.ylabel("LAI"); plt.legend(); plt.show()

✏️ **Task 2.1** · Change `bands` in the cell above and run it again. Note the test R² for each setting:

| Input bands | R² (test) |
|---|---|
| `["B8"]` | |
| `["NDVI"]` | |
| `["B4", "B5", "B8", "B11"]` | |
| all 10 bands: `["B2","B3","B4","B5","B6","B7","B8","B8A","B11","B12"]` | |

*(Double-click this cell to type your numbers into the table.)*

💬 **Questions**
1. Does it help to add more bands?
2. Look at the plot for `["NDVI"]`. Why is a straight line a bad model here?

<details>
<summary><b>Show answer</b></summary>

1. Yes. More bands carry more information, so R² goes up. Each band gets its own weight: this is multiple linear regression, a weighted sum of the bands.
2. NDVI saturates in dense canopies: at high LAI, NDVI hardly changes any more. The line is too high in the middle and too low at the end. The errors follow a pattern, which tells us the model is too simple.

</details>

## Part 3 · The same regression as a single neuron  ⏱ ~15 min

Remember the lecture: a linear regression **is** an artificial neuron without (or with a linear) activation function.
Let's prove it. We build one neuron in PyTorch and train it with gradient descent.

**Two small preparations:**
1. PyTorch works with *tensors* instead of tables. A tensor is just an array of numbers.
2. We **standardise** each input: subtract the mean, divide by the standard deviation. Then all inputs have a similar range, and gradient descent works much better.
   We compute mean and standard deviation on the training pixels only.

In [ ]:
def to_tensors(train, test, bands):
    """Standardise the input bands and convert everything to PyTorch tensors."""
    mean, std = train[bands].mean(), train[bands].std()
    X_train = torch.tensor(((train[bands] - mean) / std).values, dtype=torch.float32)
    X_test = torch.tensor(((test[bands] - mean) / std).values, dtype=torch.float32)
    y_train = torch.tensor(train[["LAI"]].values, dtype=torch.float32)
    y_test = torch.tensor(test[["LAI"]].values, dtype=torch.float32)
    return X_train, y_train, X_test, y_test

X_train, y_train, X_test, y_test = to_tensors(train, test, ["B8"])
print("X_train has shape", tuple(X_train.shape), "→ (pixels, input bands)")

Now the neuron. `nn.Linear(1, 1)` means: 1 input, 1 output. It computes exactly $\hat{y} = w \cdot x + b$.

The training loop below is the **learning loop** from the lecture:
1. **Predict** with the current weights (forward pass)
2. **Measure the error** with the loss (mean squared error)
3. **Find the blame**: backpropagation computes how each weight should change (`loss.backward()`)
4. **Adjust** the weights a little bit (gradient descent: `optimizer.step()`)

…and repeat. One round through all training pixels is one **epoch**.

In [ ]:
learning_rate = 0.1      # ✏️ try 0.001, 0.1, 0.9 and 1.1
epochs = 50

torch.manual_seed(0)
neuron = nn.Linear(1, 1)                                     # w and b start with random values
optimizer = torch.optim.SGD(neuron.parameters(), lr=learning_rate)
loss_fn = nn.MSELoss()

losses = []
for epoch in range(epochs):
    y_pred = neuron(X_train)               # 1. predict
    loss = loss_fn(y_pred, y_train)        # 2. measure the error
    optimizer.zero_grad()
    loss.backward()                        # 3. backpropagation: find the blame
    optimizer.step()                       # 4. adjust the weights
    losses.append(loss.item())

w, b = neuron.weight.item(), neuron.bias.item()
print(f"after {epochs} epochs:  w = {w:.3f},  b = {b:.3f},  training loss = {losses[-1]:.3f}")

plt.plot(losses, color=MODEL, lw=2)
plt.xlabel("epoch"); plt.ylabel("training loss (MSE)"); plt.title(f"learning rate = {learning_rate}")
plt.yscale("log"); plt.show()

Did the neuron find the same line as the linear regression? We compare with `LinearRegression` on the same standardised input:

In [ ]:
check = LinearRegression().fit(X_train.numpy(), y_train.numpy())
print(f"linear regression:  w = {check.coef_[0][0]:.3f},  b = {check.intercept_[0]:.3f}")
print(f"single neuron:      w = {w:.3f},  b = {b:.3f}")

✏️ **Task 3.1** · Change `learning_rate` in the training cell and run it again (then run the comparison cell).
Try `0.001`, `0.1`, `0.9` and `1.1`. What happens to the loss curve and to w and b?

💬 **Questions**
1. Does the neuron find the same w and b as the linear regression?
2. Describe what happens with a very small and with a too large learning rate.

<details>
<summary><b>Show answer</b></summary>

1. Yes (with a good learning rate). A single neuron without activation function is exactly a linear regression. The regression formula finds the best line directly, the neuron finds it step by step.
2. **0.001**: the steps are tiny. After 50 epochs the loss has barely moved and w, b are still far off. **0.1**: the loss drops quickly and flattens: we reached the minimum. **0.9**: the weights jump back and forth over the valley, but still arrive. **1.1**: the steps are so big that the loss grows instead of shrinking. Training *diverges*.

</details>

## Part 4 · A small neural network  ⏱ ~20 min

Now we add a **hidden layer**. To keep things readable we use two helper functions:
* `make_network(...)` builds a network with a hidden layer of your chosen size and activation function.
* `train_model(...)` runs the same learning loop as in Part 3 and records the loss on the training and the test pixels.

Instead of plain gradient descent we use **Adam**. Adam is a smarter version of gradient descent that adjusts the step size by itself. It is the standard choice for neural networks.

In [ ]:
def make_network(n_inputs, hidden=16, layers=1, activation="relu"):
    """Input → `layers` hidden layers with `hidden` neurons each → 1 output (LAI)."""
    act = {"relu": nn.ReLU, "tanh": nn.Tanh, None: nn.Identity}[activation]
    parts, size = [], n_inputs
    for _ in range(layers):
        parts += [nn.Linear(size, hidden), act()]
        size = hidden
    parts.append(nn.Linear(size, 1))            # output neuron: linear, so it can give any LAI value
    return nn.Sequential(*parts)


def train_model(model, X_train, y_train, X_val, y_val, learning_rate=0.01, epochs=500, verbose=True):
    """The learning loop from Part 3. Returns training and validation loss per epoch."""
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_fn = nn.MSELoss()
    history = {"train": [], "val": []}
    for epoch in range(epochs):
        model.train()
        loss = loss_fn(model(X_train), y_train)      # 1 + 2: predict and measure the error
        optimizer.zero_grad()
        loss.backward()                              # 3: backpropagation
        optimizer.step()                             # 4: adjust
        model.eval()
        with torch.no_grad():
            history["train"].append(loss.item())
            history["val"].append(loss_fn(model(X_val), y_val).item())
    if verbose:
        print(f"final loss  ·  training: {history['train'][-1]:.3f}   validation: {history['val'][-1]:.3f}")
    return history


def plot_history(history, title="", log=True):
    plt.plot(history["train"], color=MODEL, lw=2, label="training loss")
    plt.plot(history["val"], color=HIGHLIGHT, lw=2, label="validation loss")
    if log:
        plt.yscale("log")
    plt.xlabel("epoch"); plt.ylabel("loss (MSE)"); plt.title(title); plt.legend(); plt.show()


def predict(model, X):
    model.eval()
    with torch.no_grad():
        return model(X).numpy().ravel()

### 4a · Can a network bend the line?

We use NDVI as the only input, because then we can draw the result as a curve.
We compare one neuron (= linear regression) with a network that has one hidden layer.

In [ ]:
hidden_neurons = 16        # ✏️ try 1, 2, 4, 16, 64
activation = "relu"        # ✏️ try "relu", "tanh" and None (= no activation function)

X_train, y_train, X_test, y_test = to_tensors(train, test, ["NDVI"])

torch.manual_seed(0)
single = make_network(1, layers=0)                                  # just one output neuron
h_single = train_model(single, X_train, y_train, X_test, y_test, verbose=False)
torch.manual_seed(0)
net = make_network(1, hidden=hidden_neurons, activation=activation)
h_net = train_model(net, X_train, y_train, X_test, y_test, verbose=False)

# draw both models as curves over NDVI
ndvi = np.linspace(df.NDVI.min(), df.NDVI.max(), 200)
ndvi_std = torch.tensor(((ndvi - train.NDVI.mean()) / train.NDVI.std())[:, None], dtype=torch.float32)
plt.figure(figsize=(7, 4.5))
plt.scatter(train.NDVI, train.LAI, s=6, alpha=0.3, color=DATA, label="training pixels")
plt.plot(ndvi, predict(single, ndvi_std), color=HIGHLIGHT, lw=3, label="one neuron")
plt.plot(ndvi, predict(net, ndvi_std), color=MODEL, lw=3, label=f"network ({hidden_neurons} hidden, {activation})")
plt.ylim(-1, 8); plt.xlabel("NDVI"); plt.ylabel("LAI"); plt.legend(); plt.show()

evaluate(y_test, predict(single, X_test), "one neuron")
evaluate(y_test, predict(net, X_test), "network")

✏️ **Task 4.1** · Change `hidden_neurons` (1, 2, 4, 16, 64). How does the curve change?

✏️ **Task 4.2** · Set `hidden_neurons = 16` and `activation = None`. What happens? Then try `"tanh"`.

💬 **Questions**
1. Why can the network follow the curve, but the single neuron can't?
2. What happens without an activation function? Why?

<details>
<summary><b>Show answer</b></summary>

1. The hidden neurons have a nonlinear activation function. Each ReLU neuron adds a "bend", and the output neuron adds them up to a curve. More hidden neurons give more bends, so the curve can follow the saturation better. With 2 or 4 hidden neurons you can often see the individual bends. With only 1 hidden neuron the network can even be worse than the single neuron: one bend is not enough, and a ReLU neuron can switch off completely.
2. Without activation functions the network gives a straight line again, exactly like the single neuron. Two linear steps in a row are just one linear step. No matter how many neurons, a network without activation functions is a linear regression.

</details>

### 4b · All bands as input

Now we use all 10 bands, just like the linear regression at the end of Part 2. Does the network beat the linear regression?

In [ ]:
all_bands = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]
X_train, y_train, X_test, y_test = to_tensors(train, test, all_bands)

hidden_neurons = 32        # ✏️
layers = 2                 # ✏️ number of hidden layers
learning_rate = 0.01       # ✏️
epochs = 500               # ✏️

torch.manual_seed(0)
net = make_network(len(all_bands), hidden=hidden_neurons, layers=layers)
history = train_model(net, X_train, y_train, X_test, y_test, learning_rate, epochs)
plot_history(history, f"{layers} hidden layer(s) with {hidden_neurons} neurons")

linreg_all = LinearRegression().fit(train[all_bands], train.LAI)
evaluate(test.LAI, linreg_all.predict(test[all_bands]), "linear regression, 10 bands")
evaluate(y_test, predict(net, X_test), "neural network, 10 bands")

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, pred, name in zip(axes, [linreg_all.predict(test[all_bands]), predict(net, X_test)], ["linear regression", "neural network"]):
    ax.scatter(test.LAI, pred, s=6, alpha=0.5, color=DATA)
    ax.plot([0, 7], [0, 7], "--", color="grey")
    ax.set_xlabel("true LAI"); ax.set_title(name)
axes[0].set_ylabel("predicted LAI"); plt.tight_layout(); plt.show()

✏️ **Task 4.3** · Play with the settings: more or fewer hidden neurons, 1 or 3 layers, a learning rate of `0.1` or `0.0001`, more epochs.
Can you beat the default R²? Which setting makes things worse?

💬 **Questions**
1. Look at the left plot (linear regression). Where does it make systematic errors? Does the network fix this?
2. Is the network always better than the linear regression?

<details>
<summary><b>Show answer</b></summary>

1. The linear regression bends away from the 1:1 line at high LAI (saturation) and predicts negative LAI for some sparse pixels. The network follows the 1:1 line much better, because it can model the curved relationship.
2. No. With bad settings (e.g. a far too high or too low learning rate, or too few epochs) the network can be worse. And a better result on *these* test pixels does not yet mean it works everywhere. See Part 6.

</details>

## Part 5 · Overfitting  ⏱ ~10 min

What happens if we have **very few** training pixels but a **big** network?
We train a network with 64 hidden neurons on only 25 pixels. That network has 385 weights and biases, far more than 25 pixels can pin down.
We use the other pixels for **validation**.

In [ ]:
n_train_pixels = 25        # ✏️ try 25, 100, 300, 1000

four_bands = ["B4", "B5", "B8", "B11"]

small_train = train.sample(n_train_pixels, random_state=1)
X_small, y_small, X_val, y_val = to_tensors(small_train, test, four_bands)

torch.manual_seed(0)
big_net = make_network(len(four_bands), hidden=64, layers=1, activation="tanh")
history = train_model(big_net, X_small, y_small, X_val, y_val, learning_rate=0.01, epochs=1000)

best = int(np.argmin(history["val"]))
plt.axvline(best, color="grey", ls="--", lw=1)
plt.ylim(0, 3)
plot_history(history, f"{n_train_pixels} training pixels, 64 hidden neurons", log=False)
print(f"lowest validation loss: {history['val'][best]:.2f} after {best} epochs  →  at the end: {history['val'][-1]:.2f}")

✏️ **Task 5.1** · Change `n_train_pixels` and run again. How do the two curves change?

💬 **Questions**
1. How can you see in the plot that the network is overfitting?
2. What could you do against it?

<details>
<summary><b>Show answer</b></summary>

1. The training loss keeps going down (almost to zero), but the validation loss reaches its lowest point early (dashed line) and then goes up again. The network memorises its few training pixels instead of learning the general relationship.
2. Stop training where the validation loss is lowest (*early stopping*), use more training data, or a smaller network. With 300 or 1000 training pixels the two curves stay much closer together.

</details>

## Part 6 · Random vs. spatial validation  ⏱ ~15 min

So far we picked the test pixels at random. But in Part 1 we saw that **neighbouring pixels are very similar**.
With a random split, almost every test pixel has a "twin" right next to it in the training data.
Is our test then really honest?

Let's compare two ways to split the data:
* **Random split:** 30 % of the pixels, picked at random (as before).
* **Spatial split:** all pixels in one part of the study area. The model never sees this region during training.

In [ ]:
test_region = df.x >= 84                   # ✏️ try: df.x < 36   or   df.y >= 84

splits = {"random split": is_test, "spatial split": test_region}
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, (name, mask) in zip(axes, splits.items()):
    ax.scatter(df.x[~mask], df.y[~mask], s=6, color=DATA, label="training")
    ax.scatter(df.x[mask], df.y[mask], s=6, color=HIGHLIGHT, label="test")
    ax.set_title(name); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
axes[0].legend(loc="lower left"); plt.tight_layout(); plt.show()
print(f"test pixels  ·  random: {is_test.sum()}   spatial: {test_region.sum()}")

Now we train **the same network** on both splits and compare the test results:

In [ ]:
results = {}
for name, mask in splits.items():
    X_tr, y_tr, X_te, y_te = to_tensors(df[~mask], df[mask], all_bands)
    torch.manual_seed(0)
    model = make_network(len(all_bands), hidden=32, layers=2)
    train_model(model, X_tr, y_tr, X_te, y_te, learning_rate=0.01, epochs=500, verbose=False)
    results[name] = evaluate(y_te, predict(model, X_te), name)

plt.figure(figsize=(5, 3.5))
plt.bar(list(results), [r[1] for r in results.values()], color=[DATA, HIGHLIGHT], width=0.5)
plt.ylabel("R² on the test pixels"); plt.ylim(0, 1); plt.show()

✏️ **Task 6.1** · Try another test region in the first cell of Part 6, e.g. `df.x < 36` or `df.y >= 84`, and run both cells again.

💬 **Questions**
1. Why is the result so different, although we used the same network and the same data?
2. You want to make an LAI map for a whole region, including areas without field data. Which number would you report, and why?
3. Can you think of a similar problem with **time** instead of space?

<details>
<summary><b>Show answer</b></summary>

1. With the random split, test pixels have very similar neighbours in the training data. The model only needs to "recognise" a place it has already seen. With the spatial split, the test region is new: soil and leaf properties are a bit different there, so the model has to generalise. That is much harder, and the random split hides this.
2. The spatial result. It tells you how well the model works in places it has not seen, and that is exactly what a map of a whole region needs. The random split would make the model look better than it is.
3. Yes: if you train on images from 2023 and test on other pixels from 2023, the test is too optimistic for a map of 2025. To check how well a model works for a new year, test it on a year it has not seen.

</details>

## Wrap-up

Today you have
* fitted a **linear regression** and seen that it is the same as a **single neuron**,
* trained the neuron with **gradient descent** and seen what the **learning rate** does,
* added a **hidden layer** and seen why we need **activation functions**,
* watched a network **overfit**, and
* found out that the way you **test** a model can change the result a lot.

💬 **Final question** · In the lecture we asked: *is machine learning really "learning", or just "guessing"?* What do you think now?

<details>
<summary><b>Show answer</b></summary>

There is no single right answer. But a good one goes like this: the model does make guesses, but they are *educated* guesses. It measures how wrong it is (loss), improves step by step (gradient descent and backpropagation), and we check honestly whether its guesses also work on new data (validation). The last part is our job, not the model's.

</details>

**Next week:** so far, every pixel was handled on its own. But a pixel's neighbours tell us a lot, as you saw in Part 6.
Convolutional neural networks use exactly this spatial context.